# D7 Variant Orchestration

**Source script:** `run_eda_d7_compare.py`

Notebook mirror for submission traceability.

In [ ]:
from __future__ import annotations

import argparse
import json
import shlex
import subprocess
import sys
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Sequence

import pandas as pd


ROOT = Path(__file__).resolve().parent
DEFAULT_INPUT = ROOT / "outputs" / "imputed_dataset_enriched.csv"
DEFAULT_OUTROOT = ROOT / "outputs" / "eda_v3_d7_compare"


@dataclass(frozen=True)
class VariantSpec:
    name: str
    description: str


VARIANT_SPECS: Sequence[VariantSpec] = (
    VariantSpec("original_d7_twice", "Original dataset; all D7 rows retained."),
    VariantSpec("d7_first_only", "Keep only earliest D7 row by presentation date; remove other D7 rows."),
    VariantSpec("d7_second_only", "Keep only latest D7 row by presentation date; remove other D7 rows."),
    VariantSpec("no_d7", "Drop all D7 rows."),
)


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(description="Isolated D7 sensitivity pipeline for EDA v3 stack.")
    parser.add_argument("--input", type=Path, default=DEFAULT_INPUT, help="Base input dataset")
    parser.add_argument("--output-root", type=Path, default=DEFAULT_OUTROOT, help="Isolated output root")
    parser.add_argument("--id-col", type=str, default="ID", help="Identifier column")
    parser.add_argument("--target-id", type=str, default="D7", help="Identifier value to sensitivity-test")
    parser.add_argument("--date-col", type=str, default="", help="Optional presentation date column override")
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--corr-threshold", type=float, default=0.85)
    parser.add_argument("--cv-folds", type=int, default=5)
    parser.add_argument("--test-size", type=float, default=0.2)
    parser.add_argument("--or-bootstrap", type=int, default=250)
    parser.add_argument("--perm-n", type=int, default=200)
    parser.add_argument("--perm-jobs", type=int, default=1)
    parser.add_argument("--holdout-bootstrap-n", type=int, default=2000)
    parser.add_argument(
        "--variants",
        type=str,
        default="all",
        help="Comma-separated subset of variants to run: original_d7_twice,d7_first_only,d7_second_only,no_d7 (default: all)",
    )

    parser.add_argument("--skip-eda", action="store_true", help="Skip run_eda_v3 full pipeline stage")
    parser.add_argument("--skip-unsupervised", action="store_true", help="Skip run_eda_v3 --run-unsupervised stage")
    parser.add_argument("--skip-lunar", action="store_true", help="Skip lunar_final_analysis stage")
    parser.add_argument("--skip-apriori-final2", action="store_true", help="Skip apriori_refined_final2 stage")
    parser.add_argument("--skip-clinical-threshold", action="store_true", help="Skip clinical_threshold_apriori_extension stage")
    parser.add_argument("--skip-bun-validation", action="store_true", help="Skip bun_rule_validation stage")
    parser.add_argument("--skip-final-validation", action="store_true", help="Skip final_validation_pass stage")
    parser.add_argument("--skip-holdout-audit", action="store_true", help="Skip holdout_forensic_audit stage")

    parser.add_argument("--run-permutation-test", action="store_true", default=True, help="Enable permutation testing in run_eda_v3/final validation")
    parser.add_argument("--no-run-permutation-test", dest="run_permutation_test", action="store_false", help="Disable permutation tests")
    parser.add_argument("--run-lightgbm-replication", action="store_true", default=True, help="Enable LightGBM replication in run_eda_v3/final validation")
    parser.add_argument("--no-run-lightgbm-replication", dest="run_lightgbm_replication", action="store_false", help="Disable LightGBM replication")
    parser.add_argument("--run-shap-stability", action="store_true", default=True, help="Enable SHAP stability in run_eda_v3")
    parser.add_argument("--no-run-shap-stability", dest="run_shap_stability", action="store_false", help="Disable SHAP stability in run_eda_v3")

    parser.add_argument("--execute", action="store_true", help="Actually execute commands (default is dry-run)")
    parser.add_argument("--continue-on-error", action="store_true", help="Continue remaining stages if one stage fails")
    return parser.parse_args()


def infer_id_col(df: pd.DataFrame, preferred: str) -> str:
    if preferred in df.columns:
        return preferred
    lower_map = {c.lower(): c for c in df.columns}
    if preferred.lower() in lower_map:
        return lower_map[preferred.lower()]
    for candidate in ("id", "cat_id", "patient_id", "animal_id"):
        if candidate in lower_map:
            return lower_map[candidate]
    raise ValueError(f"Could not resolve ID column. Tried: {preferred!r} and common alternatives.")


def infer_date_col(df: pd.DataFrame, user_value: str) -> str | None:
    if user_value:
        if user_value in df.columns:
            return user_value
        lower_map = {c.lower(): c for c in df.columns}
        if user_value.lower() in lower_map:
            return lower_map[user_value.lower()]
        raise ValueError(f"Provided --date-col not found: {user_value}")
    for c in ("presentation_date", "date of presentation", "presentation date"):
        if c in df.columns:
            return c
    for c in df.columns:
        lc = c.lower()
        if "presentation" in lc and "date" in lc:
            return c
    return None


def build_variant_datasets(
    df: pd.DataFrame,
    id_col: str,
    target_id: str,
    date_col: str | None,
) -> Dict[str, pd.DataFrame]:
    id_series = df[id_col].astype(str).str.strip()
    mask_target = id_series == str(target_id)
    target_rows = df.loc[mask_target].copy()
    if target_rows.empty:
        raise ValueError(f"No rows found for target ID {target_id!r} in column {id_col!r}.")
    if len(target_rows) < 2:
        raise ValueError(
            f"Need at least 2 rows for {target_id!r}; found {len(target_rows)}. "
            "This comparison requires duplicate entries."
        )

    if date_col:
        sort_dates = pd.to_datetime(target_rows[date_col], errors="coerce")
        target_rows = target_rows.assign(__sort_date=sort_dates).sort_values(
            by=["__sort_date"], na_position="last", kind="mergesort"
        )
        ordered_indices = target_rows.index.tolist()
    else:
        ordered_indices = target_rows.index.tolist()

    first_idx = ordered_indices[0]
    second_idx = ordered_indices[-1]

    variants: Dict[str, pd.DataFrame] = {}
    variants["original_d7_twice"] = df.copy()
    variants["d7_first_only"] = df.loc[(~mask_target) | (df.index == first_idx)].copy()
    variants["d7_second_only"] = df.loc[(~mask_target) | (df.index == second_idx)].copy()
    variants["no_d7"] = df.loc[~mask_target].copy()
    return variants


def select_variant_specs(variants_arg: str) -> List[VariantSpec]:
    if variants_arg.strip().lower() == "all":
        return list(VARIANT_SPECS)
    requested = [x.strip() for x in variants_arg.split(",") if x.strip()]
    known = {s.name: s for s in VARIANT_SPECS}
    unknown = [x for x in requested if x not in known]
    if unknown:
        raise ValueError(f"Unknown variant(s): {unknown}. Valid: {list(known.keys())}")
    return [known[x] for x in requested]


def make_cmd(*parts: object) -> List[str]:
    return [str(x) for x in parts]


def run_cmd(cmd: Sequence[str], log_path: Path, execute: bool) -> int:
    log_path.parent.mkdir(parents=True, exist_ok=True)
    line = " ".join(shlex.quote(str(x)) for x in cmd)
    if not execute:
        log_path.write_text(f"[dry-run]\n{line}\n", encoding="utf-8")
        print(f"[dry-run] {line}")
        return 0

    with log_path.open("w", encoding="utf-8") as fh:
        fh.write(f"$ {line}\n\n")
        fh.flush()
        proc = subprocess.run(cmd, stdout=fh, stderr=subprocess.STDOUT, check=False)
    return int(proc.returncode)


def write_variant_manifest(out_root: Path, manifest_rows: List[Dict[str, object]]) -> None:
    manifest_df = pd.DataFrame(manifest_rows)
    manifest_df.to_csv(out_root / "variant_manifest.csv", index=False)


def read_csv_safe(path: Path) -> pd.DataFrame:
    if not path.exists():
        return pd.DataFrame()
    try:
        return pd.read_csv(path)
    except Exception:
        return pd.DataFrame()


def collect_supervised_summary(variant_name: str, variant_root: Path) -> pd.DataFrame:
    cv_path = variant_root / "eda_v3" / "tables" / "cv_binary_metric_summary.csv"
    holdout_path = variant_root / "final_validation" / "holdout_results_binary.csv"
    perm_path = variant_root / "eda_v3" / "tables" / "permtest_envonly_auc_summary.csv"
    shap_path = variant_root / "eda_v3" / "tables" / "shap_stability_summary_binary.csv"

    cv = read_csv_safe(cv_path)
    holdout = read_csv_safe(holdout_path)
    perm = read_csv_safe(perm_path)
    shap = read_csv_safe(shap_path)

    if cv.empty:
        return pd.DataFrame()

    cv = cv[cv.get("feature_block", "") == "environmental_only"].copy()
    keep_cols = [c for c in ("disease", "roc_auc_mean", "roc_auc_std", "balanced_accuracy_mean") if c in cv.columns]
    cv = cv[keep_cols].copy()

    if not holdout.empty:
        holdout = holdout[holdout.get("block", "") == "environmental_only"].copy()
        hold_keep = [c for c in ("disease", "test_auc", "test_balanced_accuracy") if c in holdout.columns]
        holdout = holdout[hold_keep]
        cv = cv.merge(holdout, on="disease", how="left")

    if not perm.empty:
        perm_keep = [c for c in ("disease", "p_value", "observed_auc") if c in perm.columns]
        perm = perm[perm_keep].rename(columns={"p_value": "perm_p_value", "observed_auc": "perm_observed_auc"})
        cv = cv.merge(perm, on="disease", how="left")

    if not shap.empty:
        shap = shap[shap.get("feature_block", "") == "environmental_only"].copy()
        shap_keep = [c for c in ("disease", "spearman_mean", "jaccard_top10_mean") if c in shap.columns]
        shap = shap[shap_keep]
        cv = cv.merge(shap, on="disease", how="left")

    cv.insert(0, "variant", variant_name)
    return cv


def collect_apriori_summary(variant_name: str, variant_root: Path) -> pd.DataFrame:
    base = variant_root / "apriori_refined_final2"
    tracks = {
        "diagnosis": base / "diagnosis_rules_stable.csv",
        "severity": base / "severity_rules_stable.csv",
        "cluster_kbest": base / "cluster_rules_kbest_stable.csv",
        "cluster_k6": base / "cluster_rules_k6_stable.csv",
    }
    rows: List[Dict[str, object]] = []
    for track, path in tracks.items():
        df = read_csv_safe(path)
        if df.empty:
            rows.append({"variant": variant_name, "track": track, "stable_rules_n": 0, "max_lift": None, "median_support": None})
            continue
        max_lift = float(df["lift"].max()) if "lift" in df.columns else None
        median_support = float(df["support"].median()) if "support" in df.columns else None
        rows.append(
            {
                "variant": variant_name,
                "track": track,
                "stable_rules_n": int(len(df)),
                "max_lift": max_lift,
                "median_support": median_support,
            }
        )
    return pd.DataFrame(rows)


def collect_lunar_summary(variant_name: str, variant_root: Path) -> pd.DataFrame:
    verdict = read_csv_safe(variant_root / "lunar_final" / "moon_verdict_table.csv")
    if verdict.empty:
        return pd.DataFrame()

    overall = verdict[verdict.get("analysis_level", "").astype(str).str.startswith("overall")].copy()
    if overall.empty:
        return pd.DataFrame()

    rows: List[Dict[str, object]] = []
    for disease, part in overall.groupby("disease"):
        if "rr" not in part.columns:
            continue
        top = part.sort_values("rr", ascending=False).head(1)
        row = top.iloc[0]
        rows.append(
            {
                "variant": variant_name,
                "disease": disease,
                "best_metric": row.get("metric"),
                "best_level": row.get("level"),
                "best_rr": row.get("rr"),
                "best_rr_ci_low": row.get("rr_ci_low"),
                "best_rr_ci_high": row.get("rr_ci_high"),
                "ci_excludes_1": row.get("ci_excludes_1"),
            }
        )
    return pd.DataFrame(rows)


def save_comparison_tables(out_root: Path, variant_names: Sequence[str]) -> None:
    sup_frames: List[pd.DataFrame] = []
    apr_frames: List[pd.DataFrame] = []
    lun_frames: List[pd.DataFrame] = []

    for v in variant_names:
        vroot = out_root / v
        sup_frames.append(collect_supervised_summary(v, vroot))
        apr_frames.append(collect_apriori_summary(v, vroot))
        lun_frames.append(collect_lunar_summary(v, vroot))

    compare_dir = out_root / "comparison"
    compare_dir.mkdir(parents=True, exist_ok=True)

    pd.concat([x for x in sup_frames if not x.empty], ignore_index=True).to_csv(
        compare_dir / "supervised_env_compare.csv", index=False
    ) if any(not x.empty for x in sup_frames) else pd.DataFrame(
        columns=["variant", "disease", "roc_auc_mean", "roc_auc_std", "balanced_accuracy_mean", "test_auc", "test_balanced_accuracy", "perm_p_value", "perm_observed_auc", "spearman_mean", "jaccard_top10_mean"]
    ).to_csv(compare_dir / "supervised_env_compare.csv", index=False)

    pd.concat([x for x in apr_frames if not x.empty], ignore_index=True).to_csv(
        compare_dir / "apriori_stable_compare.csv", index=False
    ) if any(not x.empty for x in apr_frames) else pd.DataFrame(
        columns=["variant", "track", "stable_rules_n", "max_lift", "median_support"]
    ).to_csv(compare_dir / "apriori_stable_compare.csv", index=False)

    pd.concat([x for x in lun_frames if not x.empty], ignore_index=True).to_csv(
        compare_dir / "lunar_best_rr_compare.csv", index=False
    ) if any(not x.empty for x in lun_frames) else pd.DataFrame(
        columns=["variant", "disease", "best_metric", "best_level", "best_rr", "best_rr_ci_low", "best_rr_ci_high", "ci_excludes_1"]
    ).to_csv(compare_dir / "lunar_best_rr_compare.csv", index=False)


def main() -> None:
    args = parse_args()
    if args.skip_clinical_threshold and (not args.skip_bun_validation or not args.skip_final_validation):
        raise ValueError(
            "bun_rule_validation/final_validation require clinical_threshold_extension outputs. "
            "Use --skip-bun-validation and --skip-final-validation when --skip-clinical-threshold is set."
        )
    args.output_root.mkdir(parents=True, exist_ok=True)

    df = pd.read_csv(args.input)
    id_col = infer_id_col(df, args.id_col)
    date_col = infer_date_col(df, args.date_col)

    variants = build_variant_datasets(df, id_col=id_col, target_id=args.target_id, date_col=date_col)
    selected_specs = select_variant_specs(args.variants)

    datasets_dir = args.output_root / "datasets"
    datasets_dir.mkdir(parents=True, exist_ok=True)

    manifest_rows: List[Dict[str, object]] = []
    run_plan_rows: List[Dict[str, object]] = []
    variant_names = [spec.name for spec in selected_specs]

    for spec in selected_specs:
        vdf = variants[spec.name]
        vroot = args.output_root / spec.name
        vroot.mkdir(parents=True, exist_ok=True)
        vdata = datasets_dir / f"{spec.name}.csv"
        vdf.to_csv(vdata, index=False)

        manifest_rows.append(
            {
                "variant": spec.name,
                "description": spec.description,
                "dataset_path": str(vdata),
                "rows": int(len(vdf)),
                "id_column": id_col,
                "target_id": str(args.target_id),
                "date_column_used": date_col or "",
            }
        )

        logs_dir = vroot / "logs"
        logs_dir.mkdir(parents=True, exist_ok=True)
        py = sys.executable

        commands: List[Dict[str, object]] = []

        if not args.skip_eda:
            cmd = make_cmd(
                py,
                ROOT / "run_eda_v3.py",
                "--input",
                vdata,
                "--outdir",
                vroot / "eda_v3",
                "--seed",
                args.seed,
                "--cv-folds",
                args.cv_folds,
                "--corr-threshold",
                args.corr_threshold,
                "--test-size",
                args.test_size,
                "--or-bootstrap",
                args.or_bootstrap,
                "--perm-jobs",
                args.perm_jobs,
            )
            if args.run_shap_stability:
                cmd.append("--run-shap-stability")
            else:
                cmd.append("--no-shap-stability")
            if args.run_permutation_test:
                cmd.extend(["--run-permutation-test", "--perm-n", str(args.perm_n)])
            if args.run_lightgbm_replication:
                cmd.append("--run-lightgbm-replication")
            commands.append({"stage": "eda_v3_full", "cmd": cmd, "log": logs_dir / "eda_v3_full.log"})

        if not args.skip_unsupervised:
            cmd = make_cmd(
                py,
                ROOT / "run_eda_v3.py",
                "--input",
                vdata,
                "--outdir",
                vroot / "eda_v3",
                "--seed",
                args.seed,
                "--cv-folds",
                args.cv_folds,
                "--corr-threshold",
                args.corr_threshold,
                "--run-unsupervised",
            )
            commands.append({"stage": "eda_v3_unsupervised", "cmd": cmd, "log": logs_dir / "eda_v3_unsupervised.log"})

        if not args.skip_lunar:
            cmd = make_cmd(py, ROOT / "lunar_final_analysis.py", "--input", vdata, "--outdir", vroot / "lunar_final", "--seed", args.seed)
            commands.append({"stage": "lunar_final", "cmd": cmd, "log": logs_dir / "lunar_final.log"})

        if not args.skip_apriori_final2:
            cmd = make_cmd(
                py,
                ROOT / "apriori_refined_final2.py",
                "--input",
                vdata,
                "--outdir",
                vroot / "apriori_refined_final2",
                "--seed",
                args.seed,
                "--corr-threshold",
                args.corr_threshold,
            )
            commands.append({"stage": "apriori_refined_final2", "cmd": cmd, "log": logs_dir / "apriori_refined_final2.log"})

        clinical_state_path = vroot / "clinical_threshold_extension" / "imputed_dataset_enriched_with_clinical_states.csv"
        severity_rules_path = vroot / "clinical_threshold_extension" / "severity_rules_clinical_thresholds.csv"

        if not args.skip_clinical_threshold:
            cmd = make_cmd(
                py,
                ROOT / "clinical_threshold_apriori_extension.py",
                "--input",
                vdata,
                "--outdir",
                vroot / "clinical_threshold_extension",
                "--seed",
                args.seed,
                "--corr-threshold",
                args.corr_threshold,
            )
            commands.append({"stage": "clinical_threshold_extension", "cmd": cmd, "log": logs_dir / "clinical_threshold_extension.log"})

        if not args.skip_bun_validation:
            cmd = make_cmd(
                py,
                ROOT / "bun_rule_validation.py",
                "--input",
                clinical_state_path,
                "--rules",
                severity_rules_path,
                "--outdir",
                vroot / "clinical_threshold_extension" / "bun_rule_validation",
            )
            commands.append({"stage": "bun_rule_validation", "cmd": cmd, "log": logs_dir / "bun_rule_validation.log"})

        if not args.skip_final_validation:
            cmd = make_cmd(
                py,
                ROOT / "final_validation_pass.py",
                "--input",
                vdata,
                "--clinical-state-input",
                clinical_state_path,
                "--severity-rules",
                severity_rules_path,
                "--outdir",
                vroot / "final_validation",
                "--seed",
                args.seed,
                "--corr-threshold",
                args.corr_threshold,
                "--perm-n",
                args.perm_n,
                "--perm-jobs",
                args.perm_jobs,
            )
            commands.append({"stage": "final_validation_pass", "cmd": cmd, "log": logs_dir / "final_validation_pass.log"})

        if not args.skip_holdout_audit:
            cmd = make_cmd(
                py,
                ROOT / "holdout_forensic_audit.py",
                "--input",
                vdata,
                "--outdir",
                vroot / "final_validation" / "holdout_audit",
                "--seed",
                args.seed,
                "--corr-threshold",
                args.corr_threshold,
                "--bootstrap-n",
                args.holdout_bootstrap_n,
            )
            commands.append({"stage": "holdout_forensic_audit", "cmd": cmd, "log": logs_dir / "holdout_forensic_audit.log"})

        status = "ok"
        for item in commands:
            stage = str(item["stage"])
            cmd = item["cmd"]
            log_path = item["log"]
            print(f"[{spec.name}] stage={stage}")
            rc = run_cmd(cmd, log_path=Path(log_path), execute=args.execute)
            run_plan_rows.append(
                {
                    "variant": spec.name,
                    "stage": stage,
                    "return_code": rc,
                    "log_path": str(log_path),
                    "command": " ".join(shlex.quote(x) for x in cmd),
                }
            )
            if rc != 0:
                status = "failed"
                if not args.continue_on_error:
                    break
        if status == "failed" and not args.continue_on_error:
            break

    write_variant_manifest(args.output_root, manifest_rows)
    pd.DataFrame(run_plan_rows).to_csv(args.output_root / "run_stage_status.csv", index=False)
    save_comparison_tables(args.output_root, variant_names=variant_names)

    meta = {
        "input": str(args.input),
        "output_root": str(args.output_root),
        "id_col": id_col,
        "target_id": args.target_id,
        "date_col": date_col,
        "seed": args.seed,
        "corr_threshold": args.corr_threshold,
        "cv_folds": args.cv_folds,
        "run_permutation_test": bool(args.run_permutation_test),
        "run_lightgbm_replication": bool(args.run_lightgbm_replication),
        "run_shap_stability": bool(args.run_shap_stability),
        "execute": bool(args.execute),
        "continue_on_error": bool(args.continue_on_error),
    }
    (args.output_root / "run_config.json").write_text(json.dumps(meta, indent=2), encoding="utf-8")

    print("\nDone.")
    print(f"Output root: {args.output_root}")
    print(f"Stage status: {args.output_root / 'run_stage_status.csv'}")
    print(f"Comparison tables: {args.output_root / 'comparison'}")


if __name__ == "__main__":
    main()
